# 사용자 인터페이스 설계서 자동 생성 Agent - 빠른 테스트용

입력:
- 앞 단계 Agent가 생성한 사용자 요구사항 정의서 JSON 파일: `.json`
- 번호 버튼이 포함될 UI 프로토타입 이미지 파일: `.png`, `.jpg`, `.jpeg`

출력:
- 사용자 인터페이스 설계서 DOCX
- 화면별 번호 버튼 합성 이미지
- 중간 산출물 JSON

흐름:
1. 사용자 요구사항 정의서 JSON 로드
2. 프로토타입 이미지 1장 확인
3. 요구사항 JSON과 이미지 기반 화면 상세 설계 생성
4. 처리내용 번호 버튼 합성 이미지 생성
5. 화면 구조도와 DOCX 생성


In [1]:
# 0. 설치
import importlib.util
import subprocess
import sys

REQUIRED_PACKAGES = {
    "torch": "torch",
    "PIL": "pillow",
    "docx": "python-docx",
    "pypdf": "pypdf",
    "transformers": "transformers",
    "qwen_vl_utils": "qwen-vl-utils",
    "tqdm": "tqdm",
    "accelerate": "accelerate",
    "olefile": "olefile",
    "pandas": "pandas",
}

missing = [pkg for module, pkg in REQUIRED_PACKAGES.items() if importlib.util.find_spec(module) is None]
if missing:
    print("설치가 필요한 패키지:", missing)
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-U", *missing])
else:
    print("필수 패키지가 이미 설치되어 있습니다.")


필수 패키지가 이미 설치되어 있습니다.


In [2]:
# 1. 기본 설정
import os
import re
import json
import subprocess
import zlib
import struct
import traceback
from datetime import datetime
from pathlib import Path
from typing import List, Dict, Any, Optional, Union

import torch
from PIL import Image, ImageDraw, ImageFont
from tqdm.auto import tqdm
from docx import Document
from docx.shared import Inches, Pt, Cm
from docx.enum.text import WD_ALIGN_PARAGRAPH
from docx.enum.table import WD_TABLE_ALIGNMENT, WD_CELL_VERTICAL_ALIGNMENT
from docx.oxml import OxmlElement
from docx.oxml.ns import qn

from qwen_vl_utils import process_vision_info
from transformers import AutoProcessor, AutoModelForImageTextToText
try:
    from transformers import Qwen2VLForConditionalGeneration
except ImportError:
    Qwen2VLForConditionalGeneration = None

INPUT_DIR = Path("./input")
OUTPUT_DIR = Path("./output")
WORK_DIR = Path("./work")

SUPPORTED_REQUIREMENT_JSON_EXTS = {".json"}
SUPPORTED_IMAGE_EXTS = {".png", ".jpg", ".jpeg"}

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
WORK_DIR.mkdir(parents=True, exist_ok=True)

MODEL_ID = "Qwen/Qwen2-VL-2B-Instruct"

MAX_NEW_TOKENS_SCREEN = 512
MAX_NEW_TOKENS_FINAL = 384

OUTPUT_DOCX_PATH = OUTPUT_DIR / "사용자_인터페이스_설계서_quick_test.docx"

print("INPUT_DIR:", INPUT_DIR)
print("SUPPORTED_REQUIREMENT_JSON_EXTS:", sorted(SUPPORTED_REQUIREMENT_JSON_EXTS))
print("SUPPORTED_IMAGE_EXTS:", sorted(SUPPORTED_IMAGE_EXTS))
print("OUTPUT_DOCX_PATH:", OUTPUT_DOCX_PATH)


INPUT_DIR: input
SUPPORTED_REQUIREMENT_JSON_EXTS: ['.json']
SUPPORTED_IMAGE_EXTS: ['.jpeg', '.jpg', '.png']
OUTPUT_DOCX_PATH: output\사용자_인터페이스_설계서_quick_test.docx


In [3]:
# 2. 사용자 요구사항 정의서 JSON 입력 사용

# 이 노트북은 input 폴더의 사용자 요구사항 정의서 JSON을 직접 읽습니다.
# 문서 텍스트 추출과 요구사항 JSON 생성은 앞 단계 Agent에서 수행합니다.


In [4]:
# 3. 입력 경로 및 JSON/이미지 파일 수집

def split_path_input(path_input: Optional[Union[str, Path, List[Union[str, Path]]]]) -> List[Path]:
    """문자열/Path/리스트 입력을 여러 개의 Path 목록으로 변환합니다."""
    if path_input is None:
        return []
    if isinstance(path_input, (str, Path)):
        raw_items = re.split(r"[,;\n]+", str(path_input))
    else:
        raw_items = [str(item) for item in path_input]
    return [Path(item.strip().strip('"').strip("'")) for item in raw_items if item and item.strip()]


def prompt_paths(label: str, default_dir: Path, allowed_exts: set) -> List[Path]:
    """실행 시 사용자에게 파일 또는 폴더 경로를 입력받습니다."""
    ext_text = ", ".join(sorted(allowed_exts))
    user_input = input(f"{label} 파일/폴더 경로를 입력하세요({ext_text}, 여러 개는 쉼표로 구분, 기본: {default_dir}): ").strip()
    return split_path_input(user_input or default_dir)


def collect_files(path_input: Optional[Union[str, Path, List[Union[str, Path]]]], allowed_exts: set, label: str) -> List[Path]:
    """입력받은 파일/폴더 경로에서 허용 확장자 파일을 재귀적으로 수집합니다."""
    paths = split_path_input(path_input)
    files = []

    for path in paths:
        if path.is_dir():
            files.extend(sorted(p for p in path.rglob("*") if p.is_file() and p.suffix.lower() in allowed_exts))
        elif path.is_file():
            if path.suffix.lower() not in allowed_exts:
                raise ValueError(f"지원하지 않는 {label} 파일 형식입니다: {path.name}. 허용 형식: {sorted(allowed_exts)}")
            files.append(path)
        else:
            raise FileNotFoundError(f"{label} 경로를 찾을 수 없습니다: {path}")

    unique_files = []
    seen = set()
    for file_path in files:
        resolved = file_path.resolve()
        if resolved not in seen:
            seen.add(resolved)
            unique_files.append(file_path)

    if not unique_files:
        raise RuntimeError(f"{label} 파일을 찾지 못했습니다. 허용 형식: {sorted(allowed_exts)}")

    return unique_files


def collect_requirement_json_paths(path_input: Optional[Union[str, Path, List[Union[str, Path]]]] = None) -> List[Path]:
    """사용자 요구사항 정의서 JSON 파일을 한 개 이상 수집합니다."""
    if path_input is None:
        path_input = prompt_paths("사용자 요구사항 정의서 JSON", INPUT_DIR, SUPPORTED_REQUIREMENT_JSON_EXTS)
    return collect_files(path_input, SUPPORTED_REQUIREMENT_JSON_EXTS, "사용자 요구사항 정의서 JSON")


def collect_image_paths(path_input: Optional[Union[str, Path, List[Union[str, Path]]]] = None) -> List[Path]:
    """프로토타입 이미지 파일(png, jpg, jpeg)을 한 개 이상 수집합니다."""
    if path_input is None:
        path_input = prompt_paths("프로토타입 이미지", INPUT_DIR, SUPPORTED_IMAGE_EXTS)
    return collect_files(path_input, SUPPORTED_IMAGE_EXTS, "프로토타입 이미지")



def load_requirement_summary_json(json_paths: List[Path]) -> Dict[str, Any]:
    """앞 단계 Agent가 생성한 사용자 요구사항 정의서 JSON을 읽어 화면 분석 입력으로 사용합니다."""
    loaded_items = []
    for json_path in json_paths:
        with open(json_path, "r", encoding="utf-8-sig") as f:
            data = json.load(f)
        if isinstance(data, list):
            loaded_items.extend(data)
        elif isinstance(data, dict):
            loaded_items.append(data)
        else:
            raise ValueError(f"JSON 최상위 구조는 객체 또는 배열이어야 합니다: {json_path}")

    if len(loaded_items) == 1 and isinstance(loaded_items[0], dict):
        summary = loaded_items[0]
    else:
        requirements = []
        for item in loaded_items:
            if isinstance(item, dict) and isinstance(item.get("requirements"), list):
                requirements.extend(item["requirements"])
            elif isinstance(item, dict):
                requirements.append(item)
        summary = {"requirements": requirements, "requirements_count": len(requirements)}

    if "requirements" not in summary or not isinstance(summary.get("requirements"), list):
        raise ValueError("사용자 요구사항 정의서 JSON에는 requirements 배열이 필요합니다.")

    summary.setdefault("requirements_count", len(summary.get("requirements", [])))
    summary.setdefault("system_name", summary.get("project_name", ""))
    summary.setdefault("subsystem_name", "")
    return summary


def ensure_docx_output_path(output_path: Path) -> Path:
    """출력 파일 확장자를 docx로 제한하고 저장 폴더를 준비합니다."""
    output_path = Path(output_path)
    if output_path.suffix.lower() != ".docx":
        output_path = output_path.with_suffix(".docx")
    output_path.parent.mkdir(parents=True, exist_ok=True)
    return output_path


In [5]:
# 4. Qwen2-VL 모델 로드

def load_qwen_vl(model_id: str = MODEL_ID):
    """Qwen2-VL 모델과 프로세서를 로드해 추론 준비 상태로 반환합니다."""
    print(f"Loading model: {model_id}")
    try:
        processor = AutoProcessor.from_pretrained(model_id, trust_remote_code=True)

        if torch.cuda.is_available():
            dtype = torch.bfloat16
            device_map = "auto"
        else:
            dtype = torch.float32
            device_map = None

        model_cls = Qwen2VLForConditionalGeneration or AutoModelForImageTextToText
        model = model_cls.from_pretrained(
            model_id,
            torch_dtype=dtype,
            device_map=device_map,
            trust_remote_code=True,
        )
        model.eval()
        return model, processor
    except Exception as e:
        raise RuntimeError(
            "Qwen2-VL 모델 로드에 실패했습니다. 0번 설치 셀을 먼저 실행하고, 인터넷 연결/모델 다운로드 권한/GPU 메모리를 확인하세요. "
            f"원인: {e}"
        ) from e


model, processor = load_qwen_vl()


Loading model: Qwen/Qwen2-VL-2B-Instruct


Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/729 [00:00<?, ?it/s]

In [6]:
# 5. LLM/VLM 호출 함수

def qwen_generate_text(prompt: str, max_new_tokens: int = 2048) -> str:
    """텍스트 프롬프트를 모델에 전달하고 생성 결과 문자열을 반환합니다."""
    messages = [{"role": "user", "content": [{"type": "text", "text": prompt}]}]
    text = processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = processor(text=[text], padding=True, return_tensors="pt")

    if torch.cuda.is_available():
        inputs = inputs.to(model.device)

    with torch.no_grad():
        generated_ids = model.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False)

    generated_ids_trimmed = [
        out_ids[len(in_ids):] for in_ids, out_ids in zip(inputs.input_ids, generated_ids)
    ]
    return processor.batch_decode(
        generated_ids_trimmed,
        skip_special_tokens=True,
        clean_up_tokenization_spaces=False
    )[0].strip()


def qwen_analyze_image(image_path: Path, prompt: str, max_new_tokens: int = 2048) -> str:
    """이미지와 프롬프트를 함께 모델에 전달해 화면 분석 결과를 생성합니다."""
    messages = [
        {
            "role": "user",
            "content": [
                {"type": "image", "image": str(image_path)},
                {"type": "text", "text": prompt},
            ],
        }
    ]

    text = processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    image_inputs, video_inputs = process_vision_info(messages)

    inputs = processor(
        text=[text],
        images=image_inputs,
        videos=video_inputs,
        padding=True,
        return_tensors="pt",
    )

    if torch.cuda.is_available():
        inputs = inputs.to(model.device)

    with torch.no_grad():
        generated_ids = model.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False)

    generated_ids_trimmed = [
        out_ids[len(in_ids):] for in_ids, out_ids in zip(inputs.input_ids, generated_ids)
    ]
    return processor.batch_decode(
        generated_ids_trimmed,
        skip_special_tokens=True,
        clean_up_tokenization_spaces=False
    )[0].strip()


def extract_json_from_text(text: str) -> Any:
    """모델 응답에서 JSON 객체 또는 배열 부분만 찾아 파싱합니다."""
    text = text.strip()
    text = re.sub(r"^```json\s*", "", text)
    text = re.sub(r"^```\s*", "", text)
    text = re.sub(r"\s*```$", "", text)

    first_obj = text.find("{")
    first_arr = text.find("[")
    candidates = [i for i in [first_obj, first_arr] if i != -1]
    if not candidates:
        raise ValueError("JSON 시작 문자를 찾지 못했습니다.")

    start = min(candidates)
    end_obj = text.rfind("}")
    end_arr = text.rfind("]")
    end = max(end_obj, end_arr)
    if end == -1:
        raise ValueError("JSON 종료 문자를 찾지 못했습니다.")

    return json.loads(text[start:end+1])


def parse_or_repair_json(raw: str, context: str, max_new_tokens: int = 1024) -> Any:
    """모델 출력 JSON 파싱이 실패하면 원문 의미를 유지한 채 JSON 문법만 복구합니다."""
    try:
        return extract_json_from_text(raw)
    except Exception as first_error:
        repair_prompt = f"""
다음 텍스트는 {context}로 생성된 모델 응답이지만 JSON 문법이 깨졌을 수 있다.
원문에 없는 업무 내용은 새로 만들지 말고, 원문에 있는 정보만 유지해서 유효한 JSON 하나로 복구하라.
누락된 선택 필드는 빈 문자열, 빈 배열 또는 null로 둔다.
출력이 너무 길면 원문 앞부분의 핵심 항목만 유지하고 배열 항목 수를 줄여라.
ui_requirements, requirements, ui_implications는 최대 8개만 유지하라.
마크다운, 설명, 코드블록 없이 JSON만 출력하라.

[깨진 모델 응답]
{raw[:12000]}
"""
        repaired = qwen_generate_text(repair_prompt, max_new_tokens=max(max_new_tokens, 2048))
        try:
            return extract_json_from_text(repaired)
        except Exception as second_error:
            print("JSON 복구 실패. 원본 출력 일부:")
            print(raw[:1500])
            print("JSON 복구 출력 일부:")
            print(repaired[:1500])
            raise second_error from first_error


In [7]:
# 6. 사용자 요구사항 정의서 JSON 사용

# 이 노트북은 앞 단계 Agent가 생성한 사용자 요구사항 정의서 JSON을 직접 입력으로 사용합니다.
# 문서 원문을 다시 LLM으로 요약하지 않습니다.


In [8]:
# 7. 이미지별 화면 상세 분석

UI_ELEMENT_ANALYSIS_PROMPT = """
너는 사용자 인터페이스 화면을 관찰하는 UI 분석가다.
현재 이미지만 보고 화면에 실제로 보이는 텍스트와 기능 영역을 추출하라.
요구사항이나 업무 설명은 만들지 말고, 이미지에 보이는 사실만 정리하라.

반드시 JSON으로만 출력하라. 마크다운 금지.

출력 JSON schema:
{
  "screen_name_candidates": ["string"],
  "screen_type": "string",
  "menu_path_candidates": ["string"],
  "visible_texts": ["string"],
  "functional_areas": [
    {
      "name": "string",
      "visible_texts": ["string"],
      "area_role": "string",
      "x_ratio": "number",
      "y_ratio": "number"
    }
  ]
}

작성 규칙:
- 이미지에 실제로 보이는 메뉴명, 제목, 버튼명, 카드명, 표 제목, 차트명, 상태값을 최대한 분리해서 적어라.
- functional_areas는 화면에서 설명 번호를 붙일 만한 기능 영역 단위로 나누어라.
- x_ratio, y_ratio는 각 기능 영역의 대표 위치를 이미지 왼쪽 위 기준 상대 좌표로 적어라.
- 화면에 보이지 않는 업무, 요구사항, 기능은 만들지 마라.
"""

SCREEN_DETAIL_PROMPT = """
너는 공공기관 정보시스템의 사용자 인터페이스 설계서를 작성하는 UI/UX 분석 Agent다.

아래에는 현재 프로토타입 이미지에서 추출한 UI 관찰 결과와, 그 화면과 관련성이 높은 사용자 요구사항만 선별한 목록이 있다.
현재 이미지를 다시 확인하면서 화면설계서의 "3. 화면 상세 설계"에 들어갈 기본정보와 처리 내용을 생성하라.

반드시 JSON으로만 출력하라. 마크다운 금지.

출력 JSON schema:
{
  "screen_id": "string",
  "screen_name": "string",
  "screen_type": "string",
  "menu_path": "string",
  "screen_overview": "string",
  "process_contents": [
    {
      "no": "number",
      "title": "string",
      "description": "string",
      "requirement_basis": "string"
    }
  ],
  "button_markers": [
    {
      "no": "number",
      "target_area": "string",
      "x_ratio": "number",
      "y_ratio": "number"
    }
  ]
}

작성 규칙:
- 관련 유스케이스 ID, 관련 시퀀스도 ID는 절대 생성하지 말고 JSON에도 포함하지 마라.
- 화면명은 이미지 제목, 메뉴, 파일명 맥락 중 가장 구체적인 이름으로 작성하라.
- 처리내용은 반드시 화면의 실제 UI 영역 하나와 사용자 요구사항 하나 이상을 연결해서 작성하라.
- process_contents는 기능 영역별로 작성하고, title/description/requirement_basis를 서로 다르게 구체화하라.
- description에는 사용자가 해당 영역을 조회, 선택, 입력, 실행했을 때 시스템이 수행하는 처리를 한두 문장으로 작성하라.
- requirement_basis에는 근거가 된 requirement_id와 requirement_name을 포함하라.
- 같은 화면명만 반복하거나, title/description/requirement_basis를 같은 문장으로 채우지 마라.
- process_contents의 no와 button_markers의 no는 반드시 1:1로 일치시켜라.
- 번호 버튼은 텍스트를 많이 가리지 않도록 카드 모서리, 영역 외곽, 여백 근처에 배치하라.
- 프로토타입 이미지에 없는 업무를 과도하게 만들지 마라.
- 처리내용은 화면 복잡도에 따라 4~8개를 목표로 하되, 실제 기능 영역이 적으면 더 적어도 된다.

[이미지 파일명]
{image_name}

[UI 관찰 결과]
{ui_observation}

[선별된 사용자 요구사항]
{related_requirements}
"""


def normalize_text_for_match(value: Any) -> str:
    """요구사항 필터링에 사용할 텍스트를 소문자 문자열로 정리합니다."""
    if value is None:
        return ""
    if isinstance(value, (list, tuple, set)):
        return " ".join(normalize_text_for_match(v) for v in value)
    if isinstance(value, dict):
        return " ".join(normalize_text_for_match(v) for v in value.values())
    return str(value).lower()


def extract_match_terms(text: str) -> List[str]:
    """화면 텍스트와 요구사항을 비교하기 위한 핵심 토큰을 추출합니다."""
    stopwords = {
        "string", "number", "null", "true", "false", "화면", "사용자", "시스템", "요구사항",
        "기능", "처리", "관리", "제공", "지원", "정보", "데이터", "서비스", "설계", "구현",
        "확인", "조회", "입력", "출력", "목록", "상태", "결과", "내용", "기반", "관련"
    }
    terms = re.findall(r"[가-힣A-Za-z0-9_]{2,}", text.lower())
    return [term for term in terms if term not in stopwords and len(term) >= 2]


def compact_requirement(requirement: Dict[str, Any], max_text_len: int = 420) -> Dict[str, Any]:
    """모델 입력 토큰을 줄이기 위해 요구사항의 핵심 필드만 압축합니다."""
    compact = {
        "requirement_id": requirement.get("requirement_id") or "",
        "requirement_name": requirement.get("requirement_name") or "",
        "requirement_type": requirement.get("requirement_type") or "",
        "description": requirement.get("description") or "",
        "constraints": requirement.get("constraints") or [],
        "validation_criteria": requirement.get("validation_criteria") or [],
    }
    for key in ["description"]:
        if len(str(compact[key])) > max_text_len:
            compact[key] = str(compact[key])[:max_text_len].rstrip() + "..."
    compact["constraints"] = [str(v)[:180] for v in compact["constraints"][:3]]
    compact["validation_criteria"] = [str(v)[:160] for v in compact["validation_criteria"][:3]]
    return compact


def build_screen_match_context(image_path: Path, ui_observation: Dict[str, Any]) -> str:
    """이미지 관찰 결과와 파일명을 합쳐 요구사항 매칭용 화면 문맥을 만듭니다."""
    parts = [image_path.stem]
    parts.extend(ui_observation.get("screen_name_candidates", []) or [])
    parts.extend(ui_observation.get("menu_path_candidates", []) or [])
    parts.extend(ui_observation.get("visible_texts", []) or [])
    for area in ui_observation.get("functional_areas", []) or []:
        parts.append(area.get("name", ""))
        parts.append(area.get("area_role", ""))
        parts.extend(area.get("visible_texts", []) or [])
    return normalize_text_for_match(parts)


def select_related_requirements(requirement_summary: Dict[str, Any], ui_observation: Dict[str, Any], image_path: Path, limit: int = 10) -> List[Dict[str, Any]]:
    """화면 관찰 결과와 겹치는 키워드가 많은 요구사항을 우선 선별합니다."""
    requirements = requirement_summary.get("requirements", []) or []
    screen_context = build_screen_match_context(image_path, ui_observation)
    screen_terms = set(extract_match_terms(screen_context))
    scored = []
    for order, req in enumerate(requirements):
        compact = compact_requirement(req)
        req_text = normalize_text_for_match(compact)
        req_terms = set(extract_match_terms(req_text))
        overlap = screen_terms & req_terms
        score = len(overlap) * 3
        for term in screen_terms:
            if term and term in req_text:
                score += 1
        if score > 0:
            item = dict(compact)
            item["match_score"] = score
            item["matched_terms"] = sorted(overlap)[:12]
            scored.append((score, -order, item))

    scored.sort(key=lambda row: (row[0], row[1]), reverse=True)
    selected = [item for _, _, item in scored[:limit]]
    if selected:
        return selected
    return [compact_requirement(req) for req in requirements[:limit]]


def normalize_ui_observation(data: Any) -> Dict[str, Any]:
    """이미지 1차 분석 결과가 흔들려도 후속 단계에서 쓸 수 있는 형태로 정리합니다."""
    if not isinstance(data, dict):
        data = {}
    data.setdefault("screen_name_candidates", [])
    data.setdefault("menu_path_candidates", [])
    data.setdefault("visible_texts", [])
    data.setdefault("functional_areas", [])
    if not isinstance(data["functional_areas"], list):
        data["functional_areas"] = []
    normalized_areas = []
    for area in data["functional_areas"]:
        if not isinstance(area, dict):
            continue
        normalized_areas.append({
            "name": str(area.get("name") or ""),
            "visible_texts": area.get("visible_texts") if isinstance(area.get("visible_texts"), list) else [],
            "area_role": str(area.get("area_role") or ""),
            "x_ratio": safe_ratio(area.get("x_ratio"), 0.5),
            "y_ratio": safe_ratio(area.get("y_ratio"), 0.5),
        })
    data["functional_areas"] = normalized_areas
    return data


def safe_ratio(value: Any, default: float) -> float:
    """번호 버튼 좌표를 0과 1 사이의 실수로 보정합니다."""
    try:
        ratio = float(value)
    except Exception:
        ratio = default
    return max(0.03, min(0.97, ratio))


def normalize_screen_spec(data: Any, image_path: Path, ui_observation: Dict[str, Any], idx: int) -> Dict[str, Any]:
    """모델이 생성한 화면 상세 설계 JSON을 DOCX 생성에 맞는 형태로 정리합니다."""
    if not isinstance(data, dict):
        raise ValueError("화면 상세 설계 결과가 JSON 객체가 아닙니다.")
    screen_candidates = [v for v in ui_observation.get("screen_name_candidates", []) if str(v).strip()]
    data["screen_id"] = f"UI-{idx:03d}"
    data["screen_name"] = str(data.get("screen_name") or (screen_candidates[0] if screen_candidates else image_path.stem)).strip()
    data["screen_type"] = str(data.get("screen_type") or ui_observation.get("screen_type") or "").strip()
    menu_candidates = [v for v in ui_observation.get("menu_path_candidates", []) if str(v).strip()]
    data["menu_path"] = str(data.get("menu_path") or (menu_candidates[0] if menu_candidates else data["screen_name"])).strip()
    data["screen_overview"] = str(data.get("screen_overview") or "").strip()

    process_contents = data.get("process_contents", [])
    if not isinstance(process_contents, list):
        process_contents = []
    normalized_process = []
    for pos, item in enumerate(process_contents, start=1):
        if not isinstance(item, dict):
            continue
        normalized_process.append({
            "no": pos,
            "title": str(item.get("title") or "").strip(),
            "description": str(item.get("description") or "").strip(),
            "requirement_basis": str(item.get("requirement_basis") or "").strip(),
        })
    data["process_contents"] = normalized_process

    markers = data.get("button_markers", [])
    if not isinstance(markers, list):
        markers = []
    marker_by_no = {}
    for marker in markers:
        if not isinstance(marker, dict):
            continue
        try:
            no = int(marker.get("no"))
        except Exception:
            continue
        marker_by_no[no] = {
            "no": no,
            "target_area": str(marker.get("target_area") or ""),
            "x_ratio": safe_ratio(marker.get("x_ratio"), 0.5),
            "y_ratio": safe_ratio(marker.get("y_ratio"), 0.5),
        }

    areas = ui_observation.get("functional_areas", []) or []
    normalized_markers = []
    for pos, process in enumerate(data["process_contents"], start=1):
        marker = marker_by_no.get(pos)
        if marker is None:
            area = areas[pos - 1] if pos - 1 < len(areas) else {}
            marker = {
                "no": pos,
                "target_area": area.get("name") or process.get("title") or "",
                "x_ratio": safe_ratio(area.get("x_ratio"), 0.5),
                "y_ratio": safe_ratio(area.get("y_ratio"), 0.12 + min(pos, 8) * 0.09),
            }
        marker["no"] = pos
        normalized_markers.append(marker)
    data["button_markers"] = normalized_markers
    return data


def validate_screen_spec_quality(spec: Dict[str, Any]) -> List[str]:
    """반복 출력과 부실한 처리내용을 감지합니다."""
    issues = []
    process_contents = spec.get("process_contents", []) or []
    if len(process_contents) < 3:
        issues.append("처리내용이 3개 미만입니다.")
    titles = [str(item.get("title", "")).strip() for item in process_contents]
    descriptions = [str(item.get("description", "")).strip() for item in process_contents]
    bases = [str(item.get("requirement_basis", "")).strip() for item in process_contents]
    screen_name = str(spec.get("screen_name", "")).strip()
    if titles and len(set(titles)) <= max(1, len(titles) // 3):
        issues.append("처리내용 제목 반복이 많습니다.")
    if descriptions and len(set(descriptions)) <= max(1, len(descriptions) // 3):
        issues.append("처리내용 설명 반복이 많습니다.")
    if bases and len(set(bases)) <= max(1, len(bases) // 3):
        issues.append("요구사항 근거 반복이 많습니다.")
    repeated_screen_name = sum(1 for value in titles + descriptions + bases if value == screen_name)
    if repeated_screen_name >= max(2, len(process_contents)):
        issues.append("화면명만 반복된 항목이 많습니다.")
    short_descriptions = [value for value in descriptions if len(value) < 18]
    if len(short_descriptions) >= max(2, len(descriptions) // 2):
        issues.append("처리내용 설명이 너무 짧습니다.")
    marker_nos = {int(m.get("no")) for m in spec.get("button_markers", []) or [] if isinstance(m, dict) and str(m.get("no", "")).isdigit()}
    process_nos = {int(p.get("no")) for p in process_contents if isinstance(p, dict) and str(p.get("no", "")).isdigit()}
    if process_nos != marker_nos:
        issues.append("처리내용 번호와 버튼 번호가 일치하지 않습니다.")
    return issues


def analyze_screen_image(image_path: Path, requirement_summary: Dict[str, Any], idx: int) -> Dict[str, Any]:
    """이미지 관찰, 요구사항 선별, 상세 설계 생성을 순서대로 수행합니다."""
    raw_observation = qwen_analyze_image(image_path, UI_ELEMENT_ANALYSIS_PROMPT, max_new_tokens=MAX_NEW_TOKENS_SCREEN)
    try:
        ui_observation = normalize_ui_observation(parse_or_repair_json(raw_observation, "UI 관찰 JSON", max_new_tokens=MAX_NEW_TOKENS_SCREEN))
    except Exception as e:
        print(f"UI 관찰 JSON 파싱 실패: {image_path.name}")
        print(raw_observation[:1000])
        raise RuntimeError("UI 관찰 JSON 생성에 실패했습니다. 하드코딩 fallback은 사용하지 않습니다.") from e

    related_requirements = select_related_requirements(requirement_summary, ui_observation, image_path, limit=10)
    detail_prompt = (
        SCREEN_DETAIL_PROMPT
        .replace("{image_name}", image_path.name)
        .replace("{ui_observation}", json.dumps(ui_observation, ensure_ascii=False, indent=2)[:5000])
        .replace("{related_requirements}", json.dumps(related_requirements, ensure_ascii=False, indent=2)[:7000])
    )
    raw_detail = qwen_analyze_image(image_path, detail_prompt, max_new_tokens=MAX_NEW_TOKENS_SCREEN)
    try:
        data = normalize_screen_spec(
            parse_or_repair_json(raw_detail, "화면 상세 설계 JSON", max_new_tokens=MAX_NEW_TOKENS_SCREEN),
            image_path,
            ui_observation,
            idx,
        )
    except Exception as e:
        print(f"이미지 분석 JSON 파싱 실패: {image_path.name}")
        print(raw_detail[:1000])
        raise RuntimeError("이미지 분석 JSON 생성에 실패했습니다. 하드코딩 fallback은 사용하지 않습니다.") from e

    quality_issues = validate_screen_spec_quality(data)
    if quality_issues:
        retry_prompt = detail_prompt + "\n\n[품질 검증 실패 항목]\n" + json.dumps(quality_issues, ensure_ascii=False) + "\n위 문제를 반드시 수정해서 JSON만 다시 출력하라."
        raw_retry = qwen_analyze_image(image_path, retry_prompt, max_new_tokens=MAX_NEW_TOKENS_SCREEN)
        try:
            retry_data = normalize_screen_spec(
                parse_or_repair_json(raw_retry, "화면 상세 설계 재생성 JSON", max_new_tokens=MAX_NEW_TOKENS_SCREEN),
                image_path,
                ui_observation,
                idx,
            )
            retry_issues = validate_screen_spec_quality(retry_data)
            if len(retry_issues) <= len(quality_issues):
                data = retry_data
                quality_issues = retry_issues
        except Exception as e:
            print("품질 재생성 결과를 사용하지 못했습니다:", type(e).__name__, str(e)[:300])

    data["ui_observation"] = ui_observation
    data["related_requirements"] = related_requirements
    data["quality_issues"] = quality_issues
    data["image_path"] = str(image_path)
    data["annotated_image_path"] = str(create_numbered_prototype_image(image_path, data, WORK_DIR / "numbered_images"))
    if quality_issues:
        print(f"품질 경고({image_path.name}):", quality_issues)
    return data


In [9]:
# 8. 번호 버튼이 포함된 프로토타입 이미지 생성

def clamp(value: float, min_value: float, max_value: float) -> float:
    """숫자 값이 지정한 최소/최대 범위를 벗어나지 않게 보정합니다."""
    return max(min_value, min(max_value, value))


def get_marker_font(size: int):
    """번호 버튼에 사용할 굵은 글꼴을 찾고, 없으면 기본 글꼴을 반환합니다."""
    font_candidates = [
        "C:/Windows/Fonts/arialbd.ttf",
        "C:/Windows/Fonts/malgunbd.ttf",
        "C:/Windows/Fonts/Arial.ttf",
    ]
    for font_path in font_candidates:
        if Path(font_path).exists():
            return ImageFont.truetype(font_path, size=size)
    return ImageFont.load_default()


def fallback_button_markers(process_contents: List[Dict[str, Any]]) -> List[Dict[str, Any]]:
    """모델이 좌표를 주지 않았을 때 화면 가장자리 기준의 기본 번호 버튼 좌표를 생성합니다."""
    fallback_positions = [
        (0.72, 0.08), (0.90, 0.15), (0.20, 0.18), (0.20, 0.38),
        (0.58, 0.38), (0.20, 0.78), (0.50, 0.78), (0.82, 0.78),
    ]
    markers = []
    for idx, item in enumerate(process_contents):
        x_ratio, y_ratio = fallback_positions[idx % len(fallback_positions)]
        markers.append({
            "no": item.get("no", idx + 1),
            "target_area": item.get("title", ""),
            "x_ratio": x_ratio,
            "y_ratio": y_ratio,
        })
    return markers


def normalize_button_markers(screen_spec: Dict[str, Any]) -> List[Dict[str, Any]]:
    """처리내용 번호와 매칭되는 버튼 위치 목록을 정리하고 누락된 좌표를 보완합니다."""
    process_contents = screen_spec.get("process_contents", []) or []
    process_numbers = [int(item.get("no", idx + 1)) for idx, item in enumerate(process_contents)]
    raw_markers = screen_spec.get("button_markers", []) or []
    marker_by_no = {}

    for marker in raw_markers:
        try:
            no = int(marker.get("no"))
            marker_by_no[no] = marker
        except Exception:
            continue

    fallback_markers = fallback_button_markers(process_contents)
    fallback_by_no = {int(marker["no"]): marker for marker in fallback_markers}
    normalized = []

    for no in process_numbers:
        marker = marker_by_no.get(no, fallback_by_no.get(no, {"no": no, "x_ratio": 0.08, "y_ratio": 0.12}))
        normalized.append({
            "no": no,
            "target_area": marker.get("target_area", ""),
            "x_ratio": clamp(float(marker.get("x_ratio", 0.08)), 0.03, 0.97),
            "y_ratio": clamp(float(marker.get("y_ratio", 0.10)), 0.04, 0.96),
        })
    return normalized


def draw_number_marker(draw: ImageDraw.ImageDraw, x: int, y: int, radius: int, no: int, font) -> None:
    """프로토타입 이미지 위에 파란 원형 번호 버튼을 그립니다."""
    shadow_offset = max(2, radius // 8)
    draw.ellipse(
        (x - radius + shadow_offset, y - radius + shadow_offset, x + radius + shadow_offset, y + radius + shadow_offset),
        fill=(20, 34, 60, 70),
    )
    draw.ellipse((x - radius, y - radius, x + radius, y + radius), fill=(37, 99, 235, 255), outline=(255, 255, 255, 255), width=max(3, radius // 8))
    label = str(no)
    bbox = draw.textbbox((0, 0), label, font=font)
    text_w = bbox[2] - bbox[0]
    text_h = bbox[3] - bbox[1]
    draw.text((x - text_w / 2, y - text_h / 2 - radius * 0.04), label, fill=(255, 255, 255, 255), font=font)


def create_numbered_prototype_image(image_path: Path, screen_spec: Dict[str, Any], out_dir: Path) -> Path:
    """처리내용 번호 버튼을 원본 프로토타입 이미지에 합성해 새 이미지로 저장합니다."""
    out_dir.mkdir(parents=True, exist_ok=True)
    image_path = Path(image_path)
    with Image.open(image_path).convert("RGBA") as image:
        width, height = image.size
        overlay = Image.new("RGBA", image.size, (0, 0, 0, 0))
        draw = ImageDraw.Draw(overlay)
        radius = int(clamp(min(width, height) * 0.025, 18, 34))
        font = get_marker_font(int(radius * 1.15))

        for marker in normalize_button_markers(screen_spec):
            x = int(clamp(marker["x_ratio"], radius / width, 1 - radius / width) * width)
            y = int(clamp(marker["y_ratio"], radius / height, 1 - radius / height) * height)
            draw_number_marker(draw, x, y, radius, marker["no"], font)

        result = Image.alpha_composite(image, overlay).convert("RGB")
        output_path = out_dir / f"{image_path.stem}_numbered.png"
        result.save(output_path, quality=95)
        return output_path


In [10]:
# 8. 화면 구조도 생성

STRUCTURE_PROMPT = """
너는 사용자 인터페이스 설계서의 "1. 사용자 인터페이스 구조도"를 작성하는 설계자다.

아래 화면 목록을 보고 Level 1~Level 4 구조로 메뉴/화면 계층을 정리하라.

반드시 JSON 배열로만 출력하라. 마크다운 금지.

출력 JSON schema:
[
  {
    "level1": "string",
    "level2": "string",
    "level3": "string",
    "level4": "string"
  }
]

규칙:
- Level 1은 가능한 한 전체 시스템 또는 업무 영역이다.
- Level 2는 주요 메뉴 또는 서브시스템이다.
- Level 3은 화면 그룹 또는 처리 유형이다.
- Level 4는 실제 화면명 또는 세부 기능이다.
- 화면 상세 설계의 화면명이 Level 4에 최소 1회 이상 등장하게 하라.

[화면 목록]
{screen_list}
"""


def normalize_ui_structure_data(data: Any, screen_specs: List[Dict[str, Any]]) -> List[Dict[str, str]]:
    """구조도 모델 출력이 단일 객체/배열/문자열이어도 DOCX 표에 넣을 수 있는 목록으로 정리합니다."""
    if isinstance(data, dict):
        data = [data]
    if not isinstance(data, list):
        data = []

    rows = []
    for item in data:
        if isinstance(item, dict):
            rows.append({
                "level1": str(item.get("level1", "")),
                "level2": str(item.get("level2", "")),
                "level3": str(item.get("level3", "")),
                "level4": str(item.get("level4", "")),
            })

    if rows:
        return rows

    fallback_rows = []
    for s in screen_specs:
        menu_parts = [part.strip() for part in str(s.get("menu_path") or "").split("/") if part.strip()]
        fallback_rows.append({
            "level1": menu_parts[0] if len(menu_parts) > 0 else (s.get("screen_type") or ""),
            "level2": menu_parts[1] if len(menu_parts) > 1 else "",
            "level3": menu_parts[2] if len(menu_parts) > 2 else (s.get("screen_type") or ""),
            "level4": s.get("screen_name") or s.get("screen_id") or "",
        })
    return fallback_rows


def generate_ui_structure(screen_specs: List[Dict[str, Any]]) -> List[Dict[str, str]]:
    """분석된 화면 목록을 기반으로 UI 메뉴 구조도를 생성합니다."""
    screen_list = [
        {
            "screen_id": s.get("screen_id"),
            "screen_name": s.get("screen_name"),
            "menu_path": s.get("menu_path"),
            "screen_type": s.get("screen_type")
        }
        for s in screen_specs
    ]
    prompt = STRUCTURE_PROMPT.replace(
        "{screen_list}", json.dumps(screen_list, ensure_ascii=False, indent=2)
    )
    try:
        raw = qwen_generate_text(prompt, max_new_tokens=MAX_NEW_TOKENS_FINAL)
        return normalize_ui_structure_data(parse_or_repair_json(raw, "사용자 인터페이스 구조도 JSON", max_new_tokens=MAX_NEW_TOKENS_FINAL), screen_specs)
    except Exception as e:
        print("구조도 생성 실패. 화면 목록 기반 기본 구조도를 사용합니다.")
        print(type(e).__name__, str(e)[:500])
        return normalize_ui_structure_data([], screen_specs)


In [11]:
# 9. DOCX 스타일/표 유틸

def set_cell_shading(cell, fill: str):
    """DOCX 표 셀에 배경색을 적용합니다."""
    tc_pr = cell._tc.get_or_add_tcPr()
    shd = OxmlElement('w:shd')
    shd.set(qn('w:fill'), fill)
    tc_pr.append(shd)


def set_cell_text(cell, text_value: str, bold: bool = False, size: int = 9):
    """DOCX 표 셀에 텍스트와 기본 서식을 적용합니다."""
    cell.text = ""
    p = cell.paragraphs[0]
    run = p.add_run(str(text_value) if text_value is not None else "")
    run.bold = bold
    run.font.size = Pt(size)
    p.paragraph_format.space_after = Pt(0)
    cell.vertical_alignment = WD_CELL_VERTICAL_ALIGNMENT.CENTER


def set_table_borders(table):
    """DOCX 표 전체에 회색 실선 테두리를 적용합니다."""
    tbl = table._tbl
    tblPr = tbl.tblPr
    borders = OxmlElement('w:tblBorders')
    for border_name in ['top', 'left', 'bottom', 'right', 'insideH', 'insideV']:
        border = OxmlElement(f'w:{border_name}')
        border.set(qn('w:val'), 'single')
        border.set(qn('w:sz'), '6')
        border.set(qn('w:space'), '0')
        border.set(qn('w:color'), '808080')
        borders.append(border)
    tblPr.append(borders)


def add_heading(doc: Document, text_value: str, level: int = 1):
    """DOCX 문서에 설계서용 제목 문단을 추가합니다."""
    p = doc.add_paragraph()
    run = p.add_run(text_value)
    run.bold = True
    run.font.size = Pt(12 if level == 1 else 10)
    p.paragraph_format.space_before = Pt(10)
    p.paragraph_format.space_after = Pt(6)
    return p


def add_simple_table(doc: Document, headers: List[str], rows: List[List[str]], widths: Optional[List[float]] = None):
    """헤더와 행 데이터를 받아 기본 스타일의 DOCX 표를 추가합니다."""
    table = doc.add_table(rows=1, cols=len(headers))
    table.alignment = WD_TABLE_ALIGNMENT.CENTER
    set_table_borders(table)

    hdr = table.rows[0].cells
    for i, h in enumerate(headers):
        set_cell_shading(hdr[i], "D9D9D9")
        set_cell_text(hdr[i], h, bold=True, size=8)

    for row in rows:
        cells = table.add_row().cells
        for i, value in enumerate(row):
            set_cell_text(cells[i], value, size=8)

    if widths:
        for row in table.rows:
            for i, width in enumerate(widths):
                row.cells[i].width = Cm(width)

    return table


def add_key_value_table(doc: Document, pairs: List[List[str]]):
    """화면 기본정보처럼 키-값 쌍으로 구성된 DOCX 표를 추가합니다."""
    table = doc.add_table(rows=0, cols=4)
    table.alignment = WD_TABLE_ALIGNMENT.CENTER
    set_table_borders(table)

    for row_pair in pairs:
        row = table.add_row().cells
        for i, value in enumerate(row_pair):
            set_cell_text(row[i], value, bold=(i in [0,2]), size=8)
            if i in [0,2]:
                set_cell_shading(row[i], "D9D9D9")
    return table



def add_process_contents_box(doc: Document, process_contents: List[Dict[str, Any]]):
    """화면 상세 설계 하단에 예시 양식과 같은 처리 내용 영역을 추가합니다."""
    table = doc.add_table(rows=2, cols=1)
    table.alignment = WD_TABLE_ALIGNMENT.CENTER
    set_table_borders(table)
    set_cell_shading(table.cell(0, 0), "D9D9D9")
    set_cell_text(table.cell(0, 0), "처리 내용", bold=True, size=9)

    body_cell = table.cell(1, 0)
    body_cell.text = ""
    if not process_contents:
        body_cell.paragraphs[0].add_run("- 처리 내용 없음")
        return table

    first = True
    for idx, item in enumerate(process_contents, start=1):
        marker_no = item.get("no", idx)
        title = item.get("title") or f"처리 {marker_no}"
        description = item.get("description", "")
        basis = item.get("requirement_basis", "")

        p = body_cell.paragraphs[0] if first else body_cell.add_paragraph()
        first = False
        p.add_run(f"- {marker_no}. {title}").bold = True
        p.paragraph_format.space_after = Pt(2)

        if description:
            desc_p = body_cell.add_paragraph()
            desc_p.paragraph_format.left_indent = Cm(0.5)
            desc_p.add_run(f"· [{marker_no}] {description}")
        if basis:
            basis_p = body_cell.add_paragraph()
            basis_p.paragraph_format.left_indent = Cm(0.5)
            basis_p.add_run(f"· 근거: {basis}")
    return table


In [12]:
# 10. DOCX 생성 함수

def add_screen_detail_table_with_image(doc: Document, screen_spec: Dict[str, Any]):
    """화면 기본정보와 번호 버튼 이미지를 하나의 상세 설계 표 안에 넣습니다."""
    table = doc.add_table(rows=4, cols=4)
    table.alignment = WD_TABLE_ALIGNMENT.CENTER
    set_table_borders(table)

    rows = [
        ["화면ID", screen_spec.get("screen_id", ""), "화면명", screen_spec.get("screen_name", "")],
        ["화면유형", screen_spec.get("screen_type", ""), "메뉴경로", screen_spec.get("menu_path", "")],
        ["화면개요", screen_spec.get("screen_overview", ""), "", ""],
    ]
    for r_i, row_data in enumerate(rows):
        for c_i, value in enumerate(row_data):
            set_cell_text(table.cell(r_i, c_i), value, bold=(c_i in [0, 2]), size=8)
            if c_i in [0, 2]:
                set_cell_shading(table.cell(r_i, c_i), "D9D9D9")

    table.cell(2, 1).merge(table.cell(2, 3))
    image_cell = table.cell(3, 0).merge(table.cell(3, 3))
    image_cell.text = ""
    image_path = Path(screen_spec.get("annotated_image_path") or screen_spec.get("image_path", ""))
    p = image_cell.paragraphs[0]
    p.alignment = WD_ALIGN_PARAGRAPH.CENTER
    if image_path.exists():
        try:
            p.add_run().add_picture(str(image_path), width=Inches(6.7))
        except Exception:
            p.add_run(f"[이미지 삽입 실패: {image_path.name}]")
    else:
        p.add_run(f"[이미지 없음: {image_path}]")
    return table



def build_integrated_ui_design_data(
    requirement_summary: Dict[str, Any],
    ui_structure: Union[List[Dict[str, str]], Dict[str, str]],
    screen_specs: List[Dict[str, Any]],
    output_docx_path: Path
) -> Dict[str, Any]:
    """화면설계서 생성에 필요한 데이터를 하나의 통합 JSON 구조로 조립합니다."""
    ui_structure = normalize_ui_structure_data(ui_structure, screen_specs)
    align_func = globals().get("align_button_markers_to_process_contents")
    screen_details = []
    for screen in screen_specs:
        screen_copy = dict(screen)
        if callable(align_func):
            screen_copy = align_func(screen_copy)
        screen_details.append(screen_copy)
    created_date = datetime.now().strftime("%Y-%m-%d")
    return {
        "document_header": {
            "title": "사용자 인터페이스 설계서",
            "system_name": requirement_summary.get("system_name") or requirement_summary.get("project_name") or "",
            "subsystem_name": requirement_summary.get("subsystem_name") or "",
            "phase": "설계",
            "created_date": created_date,
            "version": "",
        },
        "ui_structure": ui_structure,
        "ui_screen_list": [
            {
                "screen_id": screen.get("screen_id", ""),
                "screen_name": screen.get("screen_name", ""),
            }
            for screen in screen_details
        ],
        "screen_details": screen_details,
        "source_requirements": requirement_summary,
        "outputs": {
            "docx_path": str(output_docx_path),
            "generated_at": datetime.now().isoformat(timespec="seconds"),
        },
    }


def save_integrated_ui_design_json(
    requirement_summary: Dict[str, Any],
    ui_structure: Union[List[Dict[str, str]], Dict[str, str]],
    screen_specs: List[Dict[str, Any]],
    output_docx_path: Path,
    output_json_path: Path
) -> Path:
    """통합 화면설계서 JSON을 저장합니다."""
    output_json_path = Path(output_json_path)
    output_json_path.parent.mkdir(parents=True, exist_ok=True)
    integrated_data = build_integrated_ui_design_data(
        requirement_summary=requirement_summary,
        ui_structure=ui_structure,
        screen_specs=screen_specs,
        output_docx_path=output_docx_path,
    )
    output_json_path.write_text(
        json.dumps(integrated_data, ensure_ascii=False, indent=2),
        encoding="utf-8",
    )
    return output_json_path

def create_ui_design_docx(
    requirement_summary: Dict[str, Any],
    ui_structure: Union[List[Dict[str, str]], Dict[str, str]],
    screen_specs: List[Dict[str, Any]],
    output_path: Path
):
    """확정된 화면설계서 양식 순서에 맞춰 DOCX 파일을 생성합니다."""
    output_path = ensure_docx_output_path(output_path)
    doc = Document()

    section = doc.sections[0]
    section.top_margin = Cm(1.5)
    section.bottom_margin = Cm(1.5)
    section.left_margin = Cm(1.5)
    section.right_margin = Cm(1.5)

    styles = doc.styles
    styles['Normal'].font.name = '맑은 고딕'
    styles['Normal']._element.rPr.rFonts.set(qn('w:eastAsia'), '맑은 고딕')
    styles['Normal'].font.size = Pt(9)

    title_table = doc.add_table(rows=3, cols=5)
    title_table.alignment = WD_TABLE_ALIGNMENT.CENTER
    set_table_borders(title_table)

    title_table.cell(0, 0).merge(title_table.cell(0, 4))
    set_cell_text(title_table.cell(0, 0), "사용자 인터페이스 설계서", bold=True, size=14)
    title_table.cell(0, 0).paragraphs[0].alignment = WD_ALIGN_PARAGRAPH.CENTER

    system_name = requirement_summary.get("system_name") or requirement_summary.get("project_name") or ""
    subsystem_name = requirement_summary.get("subsystem_name") or ""

    header_rows = [
        ["시스템명", system_name, "서브시스템명", subsystem_name, ""],
        ["단계명", "설계", "작성일자", datetime.now().strftime("%Y-%m-%d"), "버전"],
    ]
    for r_i, row_data in enumerate(header_rows, start=1):
        for c_i, value in enumerate(row_data):
            set_cell_text(title_table.cell(r_i, c_i), value, bold=(c_i in [0, 2, 4]), size=8)
            if c_i in [0, 2, 4]:
                set_cell_shading(title_table.cell(r_i, c_i), "D9D9D9")

    doc.add_paragraph("")

    add_heading(doc, "1. 사용자 인터페이스 구조도", level=1)
    ui_structure = normalize_ui_structure_data(ui_structure, screen_specs)
    rows = [[item.get("level1", ""), item.get("level2", ""), item.get("level3", ""), item.get("level4", "")] for item in ui_structure]
    add_simple_table(doc, ["업무 영역\n(Level 1)", "Level 2", "Level 3", "Level 4"], rows, widths=[3.0, 3.0, 3.5, 5.5])

    add_heading(doc, "2. 사용자 인터페이스 목록", level=1)
    rows = [[s.get("screen_id", ""), s.get("screen_name", "")] for s in screen_specs]
    add_simple_table(doc, ["화면 ID", "화면명"], rows, widths=[4.0, 10.0])

    add_heading(doc, "3. 화면 상세 설계", level=1)

    for s in screen_specs:
        align_func = globals().get("align_button_markers_to_process_contents")
        if callable(align_func):
            s = align_func(s)
        doc.add_page_break()
        no_text = str(s.get("screen_id", "UI-000")).split("-")[-1]
        no = int(no_text) if no_text.isdigit() else len(doc.paragraphs)
        add_heading(doc, f"3.{no} {s.get('screen_name', '')}", level=2)
        add_screen_detail_table_with_image(doc, s)
        doc.add_paragraph("")
        add_process_contents_box(doc, s.get("process_contents", []))

    try:
        doc.save(str(output_path))
    except PermissionError:
        alt_path = output_path.with_name(f"{output_path.stem}_new{output_path.suffix}")
        print(f"기존 DOCX가 열려 있어 {alt_path.name} 파일로 저장합니다.")
        doc.save(str(alt_path))
        output_path = alt_path
    return output_path


In [13]:
# 11. 전체 파이프라인 실행

def run_ui_design_agent(
    requirement_json_paths: Optional[Union[str, Path, List[Union[str, Path]]]] = None,
    image_paths: Optional[Union[str, Path, List[Union[str, Path]]]] = None,
    output_docx_path: Path = OUTPUT_DOCX_PATH
):
    """사용자 요구사항 정의서 JSON과 프로토타입 이미지를 읽어 DOCX를 생성합니다."""
    output_docx_path = ensure_docx_output_path(output_docx_path)

    print("[1/5] 입력 파일 수집 및 사용자 요구사항 정의서 JSON 로드")
    requirement_json_file_paths = collect_requirement_json_paths(requirement_json_paths)
    image_file_paths = collect_image_paths(image_paths)[:1] # 여기서 개수 조정
    print("사용자 요구사항 정의서 JSON 수:", len(requirement_json_file_paths), [p.name for p in requirement_json_file_paths])
    print("이미지 수(빠른 테스트):", len(image_file_paths), [p.name for p in image_file_paths])

    print("[2/5] 사용자 요구사항 정의서 JSON 확인")
    requirement_summary = load_requirement_summary_json(requirement_json_file_paths)
    print("요구사항 수:", len(requirement_summary.get("requirements", [])))
    (OUTPUT_DIR / "requirement_summary.json").write_text(
        json.dumps(requirement_summary, ensure_ascii=False, indent=2),
        encoding="utf-8"
    )

    print("[3/5] 프로토타입 이미지 파일 확인")
    for image_path in image_file_paths:
        try:
            with Image.open(image_path) as img:
                img.verify()
        except Exception as e:
            raise RuntimeError(f"이미지 파일을 열 수 없습니다: {image_path}") from e

    print("[4/5] 화면 이미지 분석 및 번호 버튼 이미지 생성")
    screen_specs = []
    for idx, image_path in enumerate(tqdm(image_file_paths), start=1):
        try:
            spec = analyze_screen_image(image_path, requirement_summary, idx)
            screen_specs.append(spec)
        except Exception:
            print("분석 실패:", image_path)
            traceback.print_exc()

    if not screen_specs:
        raise RuntimeError("분석된 화면이 없습니다.")

    (OUTPUT_DIR / "screen_specs.json").write_text(
        json.dumps(screen_specs, ensure_ascii=False, indent=2),
        encoding="utf-8"
    )

    print("[5/5] 화면 구조도 생성 및 DOCX 저장")
    ui_structure = generate_ui_structure(screen_specs)
    (OUTPUT_DIR / "ui_structure.json").write_text(
        json.dumps(ui_structure, ensure_ascii=False, indent=2),
        encoding="utf-8"
    )

    integrated_json_path = save_integrated_ui_design_json(
        requirement_summary=requirement_summary,
        ui_structure=ui_structure,
        screen_specs=screen_specs,
        output_docx_path=output_docx_path,
        output_json_path=OUTPUT_DIR / "ui_design_integrated.json"
    )
    print("통합 JSON 저장:", integrated_json_path.resolve())

    output_docx_path = create_ui_design_docx(
        requirement_summary=requirement_summary,
        ui_structure=ui_structure,
        screen_specs=screen_specs,
        output_path=output_docx_path
    )

    if not Path(output_docx_path).exists():
        raise RuntimeError(f"DOCX 저장 실패: {output_docx_path}")

    print("완료:", output_docx_path.resolve())
    return output_docx_path, requirement_summary, ui_structure, screen_specs


In [14]:
# 실제 실행
# input 폴더에서 사용자 요구사항 정의서 JSON과 이미지 1장을 읽어 빠른 테스트용 docx를 생성합니다.

output_docx_path, requirement_summary, ui_structure, screen_specs = run_ui_design_agent(
    requirement_json_paths=INPUT_DIR,
    image_paths=INPUT_DIR,
    output_docx_path=OUTPUT_DOCX_PATH,
)

print("생성 완료:", output_docx_path.resolve())


[1/5] 입력 파일 수집 및 사용자 요구사항 정의서 JSON 로드
사용자 요구사항 정의서 JSON 수: 1 ['사용자 요구사항 정의서 샘플.json']
이미지 수(빠른 테스트): 1 ['01_통합_AI_포털_대시보드.png']
[2/5] 사용자 요구사항 정의서 JSON 확인
요구사항 수: 63
[3/5] 프로토타입 이미지 파일 확인
[4/5] 화면 이미지 분석 및 번호 버튼 이미지 생성


  0%|          | 0/1 [00:00<?, ?it/s]

The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


품질 경고(01_통합_AI_포털_대시보드.png): ['처리내용이 3개 미만입니다.']
[5/5] 화면 구조도 생성 및 DOCX 저장
통합 JSON 저장: C:\SKN24\final_interface\output\ui_design_integrated.json
완료: C:\SKN24\final_interface\output\사용자_인터페이스_설계서_quick_test.docx
생성 완료: C:\SKN24\final_interface\output\사용자_인터페이스_설계서_quick_test.docx
